<a href="https://colab.research.google.com/github/sries74/ComfyUI-Mono/blob/main/ComfyUI_Flux2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update
!apt-get install -y wget aria2 libgl1-mesa-glx


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,227 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [345 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,573 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,633 kB]
Hit:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InR

In [ ]:
import os

# 1. Disable tracking BEFORE any comfy commands
os.environ['COMFY_CLI_DISABLE_TRACKING'] = '1'
os.environ['COMFY_TRACKING_DISABLED'] = '1'

# 2. Install base tools
!pip install -q uv comfy-cli

# 3. Skip image_gen_aux - it's a ComfyUI custom node, not a pip package
# Install other dependencies
!pip install -q insightface onnxruntime-gpu facexlib

# 4. Setup ComfyUI & Nodes
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI

%cd /content/ComfyUI
!uv pip install --system -r requirements.txt

# Install nodes with tracking disabled (environment variable should handle this)
!comfy node registry-install ComfyUI-nunchaku

# 5. Download Optimized Models
!mkdir -p models/diffusion_models models/text_encoders models/vae models/pulid

print("Downloading Models (Optimized for 15GB VRAM)...")

!wget -q --show-progress -c https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/flux1-dev-Q4_K_M.gguf -P models/diffusion_models/
!wget -q --show-progress -c https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors -P models/text_encoders/
!wget -q --show-progress -c https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors -P models/text_encoders/
!wget -q --show-progress -c https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/ae.safetensors -O models/vae/flux_vae.safetensors
!wget -q --show-progress -c https://huggingface.co/ZHO-ZHO-ZHO/Flux-identity/resolve/main/pulid_flux_v0.9.0.safetensors -P models/pulid/

# 6. Launch
!python main.py --lowvram --disable-smart-memory --use-pytorch-cross-attention

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 11.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 18.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.2 MB/s eta 0:00:00
/content/ComfyUI
Using Python 3.12.12 environment at: /usr
Audited 27 packages in 124ms
Do you agree to enable tracking to improve the application? [y/N]: 
Extracted zip file to /content/ComfyUI/custom_nodes/ComfyUI-nunchaku
Install: pip packages

## ComfyUI-Manager: 

In [ ]:
# Install Nunchaku Backend (separate from the ComfyUI-nunchaku plugin)

# The ComfyUI-nunchaku plugin is already installed via:
# comfy node registry-install ComfyUI-nunchaku

# Now install the Nunchaku backend engine
# According to official docs, this is a separate package

print("Installing Nunchaku backend...")

# Try official installation method
!pip install nunchaku-comfyui 2>/dev/null || \
  pip install nunchaku-engine 2>/dev/null || \
  echo "Standard package names not found, trying alternative..."

# If the above fails, the backend might auto-install when you first run ComfyUI
# Or install from the official Nunchaku repository
!pip install --upgrade pip
!pip install nunchaku --extra-index-url https://pypi.nunchaku.tech/simple/ --trusted-host pypi.nunchaku.tech 2>/dev/null || \
  echo "⚠️  Custom PyPI index unavailable"

# Alternative: Install from wheel or source if provided
# Check if backend installed successfully
!python -c "try:\n    import nunchaku\n    print('✅ Nunchaku backend installed successfully')\nexcept ImportError:\n    print('⚠️  Nunchaku backend not found. Plugin will attempt auto-download on first use.')" || true

print("\nNote: If backend install failed, ComfyUI-nunchaku will attempt to")
print("automatically download the backend on first use in ComfyUI.")

In [ ]:
!pip install pinggy

In [ ]:
import pinggy

tunnel1 = pinggy.start_tunnel(forwardto="localhost:8188")
print(f"Tunnel1 started - URLs: {tunnel1.urls}")

In [ ]:
!python main.py --listen 0.0.0.0
